# Request a SO-ARM101 (LeRobot)

Provision a Chameleon Edge device for SO-ARM101 robot arm teleoperation and
data collection using HuggingFace LeRobot.

## Prerequisites
- Active Chameleon Cloud allocation (project `CHI-220971`)
- A SO-ARM101 device registered on CHI@Edge
- Container image with LeRobot + teleoperation stack pushed to Docker Hub
- python-chi 1.0+ (`pip install python-chi`)

In [ ]:
import chi
from datetime import timedelta

chi.use_site("CHI@Edge")
chi.set("project_name", "CHI-261589")

from chi.lease import Lease
from chi.container import Container

## Lease the device

Request (or reuse) a lease for the SO-ARM101 edge device using the
python-chi 1.0+ `Lease` class. Update `device_name` below to match the
registered Chameleon device name.

In [ ]:
LEASE_NAME = "lerobot-soarm101-lease"
DEVICE_NAME = "soarm101-1"  # TODO: update to match your registered device

my_lease = Lease(name=LEASE_NAME, duration=timedelta(days=2))
my_lease.add_device_reservation(device_name=DEVICE_NAME, amount=1)
my_lease.submit(wait_for_active=True, idempotent=True)

print(f"Lease '{my_lease.name}' is {my_lease.status} (id: {my_lease.id})")

In [ ]:
# Re-fetch lease info (useful if reconnecting to an existing session)
my_lease = Lease.from_existing(LEASE_NAME)
print(f"Lease '{my_lease.name}' is {my_lease.status} (id: {my_lease.id})")
print(f"Device reservations: {my_lease.device_reservations}")

## Launch the container

Create a container with the LeRobot stack using the python-chi 1.0+
`Container` class. Device profiles expose USB serial (for the arm servos)
and cameras.

In [ ]:
CONTAINER_NAME = "lerobot-soarm101-container"

reservation_id = my_lease.device_reservations[0]["id"]

my_container = Container(
    name=CONTAINER_NAME,
    image_ref="rianders/lerobot-soarm101:main",  # TODO: update image tag as needed
    exposed_ports=["22/tcp"],
    reservation_id=reservation_id,
    device_profiles=[
        "pi_serial",   # USB serial access for SO-ARM101 servos
        "pi_gpio",     # GPIO access if needed
    ],
)
my_container.submit(wait_for_active=True, idempotent=True)

print(f"Container '{my_container.name}' is {my_container.status}")

In [ ]:
print(my_container.logs())

In [ ]:
print(f"Container '{my_container.name}' is {my_container.status}")

## Assign a floating IP

In [ ]:
my_container.associate_floating_ip()
print(f"Public IP: {my_container.floating_ip}")
print(f"\nSSH: ssh root@{my_container.floating_ip}")

## Verify the environment

Quick checks that the arm, cameras, and LeRobot are accessible.

In [ ]:
output, exit_code = my_container.execute("ls -l")
print(output)

In [ ]:
# Check that the SO-ARM101 serial device is visible
output, _ = my_container.execute("ls -l /dev/ttyUSB* /dev/ttyACM* 2>/dev/null || echo 'No serial devices found'")
print(output)

In [ ]:
# Check for connected cameras
output, _ = my_container.execute("ls -l /dev/video* 2>/dev/null || echo 'No video devices found'")
print(output)

In [ ]:
# Verify Python and LeRobot are installed
output, _ = my_container.execute("python3 --version && python3 -c 'import lerobot; print(f\"LeRobot version: {lerobot.__version__}\")'")
print(output)

## Teleoperation & Data Collection

Use the LeRobot v0.4.1 CLI (`lerobot-record`) to control the arm and
record demonstration episodes into a LeRobot-compatible dataset.

### Supported policies for training later
- **ACT** — recommended for MI100 (no Flash Attention needed)
- **Diffusion** — heavier compute, solid results
- **VQ-BeT** — good for multi-modal action distributions
- **Pi0Fast / Pi0.5** — VLA models, need gradient checkpointing on MI100
- **SmolVLA** — lightweight VLA for constrained hardware

In [ ]:
# Teleoperate the arm (no recording, just control)
# TODO: update camera config to match your setup
teleop_cmd = """\
lerobot-teleoperate \
    --robot.type=so101_follower \
    --robot.cameras='{ top: {type: opencv, index_or_path: 0, width: 640, height: 480, fps: 30} }' \
    --teleop.type=so101_leader
"""
print("Starting teleoperation ...")
output, _ = my_container.execute(f"bash -c '{teleop_cmd.strip()} > /tmp/teleop.log 2>&1 &'")
print(output)

In [ ]:
# Record demonstration episodes to a LeRobot dataset
# TODO: adjust repo_id, num_episodes, and camera config
record_cmd = """\
lerobot-record \
    --robot.type=so101_follower \
    --robot.cameras='{ top: {type: opencv, index_or_path: 0, width: 640, height: 480, fps: 30} }' \
    --teleop.type=so101_leader \
    --dataset.repo_id=rianders/soarm101-episodes \
    --dataset.num_episodes=10 \
    --dataset.push_to_hub=true
"""
print("Starting data collection ...")
output, _ = my_container.execute(f"bash -c '{record_cmd.strip()}'")
print(output)

In [ ]:
# List collected dataset files
output, _ = my_container.execute("find /lerobot/data -type f | head -30")
print(output)

## Cleanup

Destroy the container and release the lease when finished.

In [ ]:
# Uncomment to destroy container
# my_container.delete()

In [ ]:
# Remove lease when you're done with the device
# my_lease.delete()